# Stage 4-2. Trainer and Evaluator

이 노트북은 `src/core/trainer.py`와 `src/core/evaluator.py`를 실습한다.

| 클래스 | 메서드 | 역할 |
|---|---|---|
| `Trainer` | `fit(train_loader)` | 1 epoch 학습 루프 실행 → `{loss, metric, num_samples}` 반환 |
| `Evaluator` | `evaluate(test_loader)` | 1 pass 평가 루프 실행 → `{loss, metric, num_samples}` 반환 |

두 클래스 모두 `task_spec`을 수신하여 task별 loss/metric 함수를 자동으로 선택한다.

**학습 목표**
1. `Trainer.fit()`이 `DataLoader`를 순회하며 배치 단위로 학습하는 흐름을 확인한다.
2. `Evaluator.evaluate()`가 gradient 없이 평가만 수행하는 흐름을 확인한다.
3. 3종 task에 대해 동일 인터페이스로 학습·평가 결과를 비교한다.
4. epoch별 log를 수집하고 시각화한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.data.mnist import MnistDataset
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.core.optimizers import SGD
from src.core.trainer import Trainer
from src.core.evaluator import Evaluator
from src.task import get_task_spec

## 1. 데이터 준비

In [ ]:
DATASET_DIR = "/mnt/d/datasets/mnist"
BATCH_SIZE  = 64

train_dataset = MnistDataset("train", "multiclass", dataset_dir=DATASET_DIR)
test_dataset  = MnistDataset("test",  "multiclass", dataset_dir=DATASET_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 샘플 수  : {len(train_dataset):,}")
print(f"테스트 샘플 수: {len(test_dataset):,}")
print(f"학습 배치 수  : {len(train_loader)}")
print(f"테스트 배치 수: {len(test_loader)}")

## 2. Trainer.fit() — 1 epoch

In [ ]:
task_spec = get_task_spec("multiclass")
model     = MLP(task="multiclass", seed=42)
optimizer = SGD(model, lr=0.01)
trainer   = Trainer(model, optimizer, task_spec)

print("Trainer 설정:")
print(f"  task      : {task_spec['task']}")
print(f"  loss_fn   : {trainer.loss_fn.__name__}")
print(f"  metric_fn : {trainer.metric_fn.__name__}")
print()

train_log = trainer.fit(train_loader)
print("fit() 반환값:")
for k, v in train_log.items():
    print(f"  {k}: {v}")

## 3. Evaluator.evaluate() — 1 pass

In [ ]:
evaluator = Evaluator(model, task_spec)

print("Evaluator 설정:")
print(f"  task      : {task_spec['task']}")
print(f"  loss_fn   : {evaluator.loss_fn.__name__}")
print(f"  metric_fn : {evaluator.metric_fn.__name__}")
print()

test_log = evaluator.evaluate(test_loader)
print("evaluate() 반환값:")
for k, v in test_log.items():
    print(f"  {k}: {v}")

> **Trainer vs Evaluator**: Trainer는 `backward` + `optimizer.step()`을 실행한다. Evaluator는 `forward`만 실행하므로 gradient가 계산되지 않는다.

## 4. epoch 반복 — train/test log 수집

In [ ]:
NUM_EPOCHS = 5

model2    = MLP(task="multiclass", seed=42)
opt2      = SGD(model2, lr=0.01)
trainer2  = Trainer(model2, opt2, task_spec)
evaluator2= Evaluator(model2, task_spec)

logs = []
for epoch in range(1, NUM_EPOCHS + 1):
    tr = trainer2.fit(train_loader)
    te = evaluator2.evaluate(test_loader)
    log = {"epoch": epoch, "train": tr, "test": te}
    logs.append(log)
    print(f"Epoch {epoch:2d} | "
          f"train loss={tr['loss']:.4f} acc={tr['metric']:.4f} | "
          f"test  loss={te['loss']:.4f} acc={te['metric']:.4f}")

## 5. 학습 곡선 시각화

In [ ]:
epochs     = [l["epoch"]           for l in logs]
train_loss = [l["train"]["loss"]   for l in logs]
test_loss  = [l["test"]["loss"]    for l in logs]
train_acc  = [l["train"]["metric"] for l in logs]
test_acc   = [l["test"]["metric"]  for l in logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Multiclass MLP — Train vs Test", fontsize=13, fontweight="bold")

ax1.plot(epochs, train_loss, label="train", color="steelblue",   linewidth=2, marker="o")
ax1.plot(epochs, test_loss,  label="test",  color="#E87B4C",     linewidth=2, marker="o", linestyle="--")
ax1.set_title("Cross-Entropy Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_acc, label="train", color="steelblue",   linewidth=2, marker="o")
ax2.plot(epochs, test_acc,  label="test",  color="#E87B4C",     linewidth=2, marker="o", linestyle="--")
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 3종 task 비교 — Trainer / Evaluator

In [ ]:
metric_labels = {
    "multiclass": "Accuracy",
    "binary":     "Binary Acc",
    "regression": "R²",
}

colors = {"multiclass": "steelblue", "binary": "#E87B4C", "regression": "mediumseagreen"}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("3 Tasks — Trainer / Evaluator (3 epochs)", fontsize=13, fontweight="bold")

EPOCHS_CMP = 3

for col, task in enumerate(["multiclass", "binary", "regression"]):
    spec_cmp   = get_task_spec(task)
    tr_ds = MnistDataset("train", task, dataset_dir=DATASET_DIR)
    te_ds = MnistDataset("test",  task, dataset_dir=DATASET_DIR)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    te_dl = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False)

    mdl = MLP(task=task, seed=42)
    opt = SGD(mdl, lr=0.01)
    trnr = Trainer(mdl, opt, spec_cmp)
    evlr = Evaluator(mdl, spec_cmp)

    ep_list, tr_loss_l, te_loss_l, tr_met_l, te_met_l = [], [], [], [], []
    for ep in range(1, EPOCHS_CMP + 1):
        tr_l = trnr.fit(tr_dl)
        te_l = evlr.evaluate(te_dl)
        ep_list.append(ep)
        tr_loss_l.append(tr_l["loss"])
        te_loss_l.append(te_l["loss"])
        tr_met_l.append(tr_l["metric"])
        te_met_l.append(te_l["metric"])

    c = colors[task]
    axes[0, col].plot(ep_list, tr_loss_l, label="train", color=c,    linewidth=2, marker="o")
    axes[0, col].plot(ep_list, te_loss_l, label="test",  color=c,    linewidth=2, marker="o", linestyle="--", alpha=0.6)
    axes[0, col].set_title(f"{task} — Loss")
    axes[0, col].set_xlabel("epoch")
    axes[0, col].set_ylabel("loss")
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(ep_list, tr_met_l, label="train", color=c,    linewidth=2, marker="o")
    axes[1, col].plot(ep_list, te_met_l, label="test",  color=c,    linewidth=2, marker="o", linestyle="--", alpha=0.6)
    axes[1, col].set_title(f"{task} — {metric_labels[task]}")
    axes[1, col].set_xlabel("epoch")
    axes[1, col].set_ylabel(metric_labels[task])
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 정리

```
Trainer.fit(train_loader)
  for x, y in train_loader:
      logits = model.forward(x)
      loss   = loss_fn(logits, y)
      grad   = grad_fn(logits, y)
      model.backward(grad)
      optimizer.step()
  → {"loss", "metric", "num_samples"}

Evaluator.evaluate(test_loader)
  for x, y in test_loader:
      logits = model.forward(x)   # backward 없음
      loss   = loss_fn(logits, y)
  → {"loss", "metric", "num_samples"}
```

| 항목 | Trainer | Evaluator |
|---|---|---|
| backward | O | X |
| optimizer.step | O | X |
| 반환 키 | loss, metric, num_samples | loss, metric, num_samples |

**핵심 설계 원칙**
- `task_spec`을 수신하므로 task 변경 시 Trainer/Evaluator 코드를 수정할 필요가 없다.
- `num_samples` 기반 가중 평균으로 배치 크기가 다른 마지막 배치도 정확히 집계한다.
- Stage 4-3의 `Experiment`는 이 두 클래스를 조립하여 multi-epoch 루프를 제공한다.